# 16. Frozen feature combinations: 30 vs 60-day training windows

Evaluate FULL and the three Notebook 15 feature combinations with 60-day rolling training windows. Lists are frozen: no new feature selection. The 30-day reference comes from the saved Notebook 15 archive; input hash, regions, dates and outcomes must match exactly. All methods use 585 regions, weekly refits and direct horizons 1/2/3.

Report early-2021 selection-period fit performance separately from July-December validation. Later-validation data appeared in earlier work, so this is exploratory stability assessment, not untouched final test evidence. Do not claim the 30-day-selected lists are necessarily optimal for 60 days. Forward and Stepwise have identical lists and are one predictive specification here.


In [1]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)



GROUPS = {
    'core_history': ['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
    'rolling_level': ['cases_mean_7d','cases_mean_14d'],
    'volatility': ['case_std_7d','case_std_14d'],
    'dynamics': ['case_acceleration','case_jerk'],
    'local_rates': ['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
    'sparse_history': ['days_since_last_observed_case_censored','no_previous_observed_case'],
    'pressure': ['nj_covid_pressure_7d','nj_covid_pressure_14d'],
    'month': [f'target_month_{k}' for k in range(2,13)],
}
WEEKDAY=[f'target_weekday_{k}' for k in range(2,8)]

def independent_columns(x):
    # Training-only rank filtering. Removes algebraic duplicates and calendar
    # categories absent from the current window; not predictive feature selection.
    z=x-x.mean(axis=1,keepdims=True)
    norms=np.sqrt(np.mean(z*z,axis=(0,1)))
    selected=[]; q=[]
    for j in range(z.shape[2]):
        if norms[j]<1e-10: continue
        v=(z[:,:,j]/norms[j]).reshape(-1).copy()
        for _ in range(2):
            for u in q:v-=u*(u@v)
        length=np.linalg.norm(v)
        if length>1e-7*np.sqrt(v.size):
            selected.append(j);q.append(v/length)
    return selected


def run_frozen_window_comparison(source,output,selection_file,window=60,bootstrap_repetitions=2000):
    """Greedy backward selection on early 2021; later 2021 checked after freezing."""
    self_checks()
    source=Path(source)
    folder=Path(output)/('frozen_window_60_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    numeric=sum([v for k,v in GROUPS.items() if k!='month'],[])
    full=numeric+GROUPS['month']+WEEKDAY
    cols=sorted(set(numeric)-set(GROUPS['sparse_history']))
    p=pd.read_csv(source,usecols=cols+['zip','date','population','days_since_last_observed_case','pressure_definition'],dtype={'zip':'string'},parse_dates=['date'])
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    assert not p.duplicated(['zip','date']).any()
    ids=np.array(sorted(p.zip.unique()));dates=pd.date_range(p.date.min(),p.date.max())
    def matrix(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    population=matrix('population')[:,0]
    eligible=(population>0)&np.isfinite(population)&np.isfinite(matrix('nj_covid_pressure_14d')[:,13])
    matrices={c:matrix(c)[eligible] for c in cols}
    gap=matrix('days_since_last_observed_case')[eligible]
    matrices['no_previous_observed_case']=(~np.isfinite(gap)).astype(float)
    matrices['days_since_last_observed_case_censored']=np.where(np.isfinite(gap),gap,np.arange(len(dates))[None,:]+1)
    ids=ids[eligible];n=len(ids);counts=matrices['cases_today']
    origins=np.flatnonzero((dates>=pd.Timestamp('2021-01-01'))&((dates+pd.Timedelta(days=3))<=pd.Timestamp('2021-12-31')))
    v=len(origins);actual=np.stack([counts[:,origins+h] for h in (1,2,3)])
    selection_mask=(dates[origins]+pd.Timedelta(days=3))<=pd.Timestamp('2021-06-30')
    assessment_mask=dates[origins]>=pd.Timestamp('2021-07-01')
    assert not (selection_mask&assessment_mask).any()
    # Cached sufficient statistics let every candidate refit the actual OLS
    # on its own feature subset without repeating full raw training-data scans.
    cache=[];timing=[]
    for h in (1,2,3):
        target=dates+pd.Timedelta(days=h)
        for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),(n,len(dates)))
        for k in range(2,13):matrices[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),(n,len(dates)))
        x=np.stack([matrices[c] for c in full],axis=2)
        blocks=[]
        for first in range(0,v,7):
            block=origins[first:first+7];fit=int(block[0]);last=fit-h;train=np.arange(last-window+1,last+1)
            assert train[0]>=14 and train[-1]+h<=fit
            xt=x[:,train];yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(x[:,block]).all()
            xm=xt.mean(axis=1);ym=yt.mean(axis=1);xc=xt-xm[:,None];yc=yt-ym[:,None]
            scale=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(scale>1e-10,scale,1)
            z=xc/scale;zf=z.reshape(-1,len(full))
            gram=zf.T@zf;rhs=zf.T@yc.reshape(-1)
            blocks.append((first,len(block),gram,rhs,scale,xm,ym,x[:,block]))
            timing.append({'horizon':h,'fit_date':dates[fit],'training_origin_start':dates[train[0]],'training_origin_end':dates[train[-1]],'latest_training_target':dates[train[-1]+h]})
        cache.append(blocks)
    np.seterr(invalid='raise')
    def predict(features,return_rank_log=False):
        ix=[full.index(f) for f in features]
        predictions=np.empty((3,n,v));rank_log=[]
        for hh,blocks in enumerate(cache):
            for first,length,g,rhs,scale,xm,ym,xp in blocks:
                # Ordered Gram-Schmidt in training Gram space removes only
                # nonidentified columns. Standardization is training-only.
                selected=[]
                for j in ix:
                    if g[j,j]<1e-8:continue
                    if not selected:residual=g[j,j]
                    else:
                        gg=g[np.ix_(selected,selected)];cross=g[selected,j]
                        residual=g[j,j]-cross@np.linalg.solve(gg,cross)
                    if residual>1e-8*max(g[j,j],1):selected.append(j)
                if selected:
                    b=np.linalg.solve(g[np.ix_(selected,selected)],rhs[selected])/scale[selected]
                    a=ym-xm[:,selected]@b
                    predictions[hh,:,first:first+length]=np.einsum('ntp,p->nt',xp[:,:,selected],b)+a[:,None]
                else:predictions[hh,:,first:first+length]=ym[:,None]
                if return_rank_log:rank_log.append({'horizon':hh+1,'forecast_origin':dates[origins[first]],'estimated_slopes':len(selected),'omitted':';'.join(full[j] for j in ix if j not in selected)})
        assert np.isfinite(predictions).all()
        return (predictions,rank_log) if return_rank_log else predictions
    def losses(pred,mask):
        e=np.maximum(pred,0)[:,:,mask]-actual[:,:,mask]
        return np.abs(e).mean(axis=1),(e*e).mean(axis=1),e.mean(axis=(1,2))
    # Numerical regression check: one subset must match direct within OLS.
    hh=0;first,length,g,rhs,scale,xm,ym,xp=cache[hh][0]
    check_features=['cases_today','cases_lag_1','cases_lag_7']+WEEKDAY
    check_ix=[full.index(f) for f in check_features]
    train_end=int(origins[0])-1;tr=np.arange(train_end-window+1,train_end+1)
    check_x=np.stack([matrices[f] if f not in WEEKDAY else np.broadcast_to(((dates+pd.Timedelta(days=1)).dayofweek==int(f.rsplit('_',1)[1])-1).astype(float),(n,len(dates))) for f in check_features],axis=2)
    direct_b,direct_a,*_=fit_fe(check_x[:,tr],counts[:,tr+1])
    direct=np.einsum('ntp,p->nt',check_x[:,origins[:length]],direct_b)+direct_a[:,None]
    assert np.allclose(predict(check_features)[0,:,:length],direct,atol=1e-7,rtol=1e-7)


    selection_file=Path(selection_file)
    selection=json.loads(selection_file.read_text(encoding='utf-8'))
    original_protocol=json.loads((selection_file.parent/'protocol.json').read_text(encoding='utf-8'))
    source_hash=hashlib.sha256(source.read_bytes()).hexdigest()
    assert source_hash==original_protocol['source_sha256'],'Input panel differs from feature-selection run'
    old_archive=selection_file.parent/'final_predictions.npz'
    with np.load(old_archive,allow_pickle=False) as archive:
        assert np.array_equal(ids,archive['zip']) and np.array_equal(dates[origins].to_numpy(),archive['date'])
        assert np.array_equal(actual,archive['actual_cases'])
        assert np.array_equal(selection_mask,archive['selection_mask']) and np.array_equal(assessment_mask,archive['assessment_mask'])
        prior_predictions=archive['raw_predictions'].copy()
        models=list(archive['models'])
    assert models==['FULL','BACKWARD','FORWARD','STEPWISE']
    assert window==60,'This run is the prespecified frozen-list 60-day sensitivity'
    protocol={'source':str(source.resolve()),'source_sha256':source_hash,'frozen_selection_file':str(selection_file.resolve()),
        'frozen_selection_sha256':hashlib.sha256(selection_file.read_bytes()).hexdigest(),'window':window,
        'reference_window':30,'regions':n,'horizons':[1,2,3],'retrain_every_days':7,
        'features_reselected':False,'feature_lists_frozen':selection,'reference_prediction_archive':str(old_archive.resolve()),
        'periods':{'selection':'January-June 2021, historical selection period','later_validation':'July-December 2021, temporal stability check'},
        'scope':'Frozen 30-day-selected feature combinations evaluated with 60-day parameter windows. Does not determine features optimal for 60 days.',
        'inference':'Exploratory date-level HAC with 14-day Bartlett bandwidth and pointwise 2000 moving-date-block intervals. Holm across this run comparisons. Validation has been examined before; no untouched-test claim.',
        'test_2022_2023_evaluated':False}
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    pd.DataFrame({'zip':ids}).to_csv(folder/'common_regions.csv',index=False)
    pd.DataFrame(timing).to_csv(folder/'training_timing.csv',index=False)
    predictions=[];rank_logs=[]
    for method in models:
        features=full if method=='FULL' else selection[method]['selected_columns']
        pred,rank_log=predict(features,True)
        predictions.append(pred)
        for row in rank_log:row['method']=method
        rank_logs.extend(rank_log)
        print(f'60-day frozen specification {method} fitted ({len(features)} candidate columns).',flush=True)
    predictions=np.stack(predictions)
    assert np.allclose(predictions[2],predictions[3],atol=1e-9)
    metrics=[];daily=[];paired=[];period_scores={}
    def contrast(period,baseline,candidate,a,b,label):
        d=(a-b).mean(axis=0);delta,se,pv=hac_mean_test(d,14)
        draws=block_indices(len(d),14,bootstrap_repetitions,20261101)
        lo,hi=np.quantile(d[draws].mean(axis=1),[.025,.975])
        paired.append({'period':period,'comparison_type':label,'baseline':baseline,'candidate':candidate,
            'delta_MAE_baseline_minus_candidate':delta,'ci95_low_pointwise':lo,'ci95_high_pointwise':hi,
            'p_HAC_raw_exploratory':pv,'interpretation':'Positive favors candidate; p-values do not measure practical benefit.'})
    for period,mask in [('selection',selection_mask),('later_validation',assessment_mask)]:
        for w,allpred in [(30,prior_predictions),(60,predictions)]:
            for m,method in enumerate(models):
                ae,mse,bias=losses(allpred[m],mask)
                period_scores[(period,w,method)]=(ae,mse)
                for h in range(3):
                    metrics.append({'period':period,'window':w,'method':method,'horizon':h+1,
                        'feature_columns':len(full) if method=='FULL' else selection[method]['feature_columns'],
                        'regions':n,'forecast_dates':int(mask.sum()),'MAE':float(ae[h].mean()),'RMSE':float(np.sqrt(mse[h].mean())),'bias':float(bias[h])})
                    for day,a,ms in zip(dates[origins[mask]],ae[h],mse[h]):daily.append({'period':period,'window':w,'method':method,'horizon':h+1,'date':day,'MAE':a,'MSE':ms})
        for ia,baseline in enumerate(models):
            for candidate in models[ia+1:]:
                contrast(period,baseline,candidate,period_scores[(period,60,baseline)][0],period_scores[(period,60,candidate)][0],'methods_at_60_days')
        for method in models:
            contrast(period,f'{method}_30d',f'{method}_60d',period_scores[(period,30,method)][0],period_scores[(period,60,method)][0],'same_frozen_features_30_vs_60')
    metrics=pd.DataFrame(metrics)
    summary=metrics.groupby(['period','window','method','feature_columns'],as_index=False)[['MAE','RMSE','bias']].mean()
    summary.to_csv(folder/'window_method_comparison_summary.csv',index=False)
    metrics.to_csv(folder/'window_method_comparison_by_horizon.csv',index=False)
    paired=pd.DataFrame(paired);paired['p_Holm_all_20_exploratory']=holm_adjust(paired.p_HAC_raw_exploratory)
    paired.to_csv(folder/'paired_comparisons.csv',index=False)
    pd.DataFrame(daily).to_csv(folder/'daily_losses.csv',index=False)
    pd.DataFrame(rank_logs).to_csv(folder/'rank_logs_60d.csv',index=False)
    np.savez_compressed(folder/'predictions_60d.npz',raw_predictions=predictions,actual_cases=actual,models=np.array(models),zip=ids,date=dates[origins].to_numpy(),horizons=np.array([1,2,3]),selection_mask=selection_mask,assessment_mask=assessment_mask)
    (folder/'frozen_feature_lists.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
    report='# Frozen feature combinations: 30 vs 60-day windows\n\n'
    report+='Feature membership is taken unchanged from the three-method Notebook 15 run. All 585 regions, forecast origins and horizons match the 30-day comparison. Only the rolling parameter training window changes to 60 days. Weekly refits still use only outcomes dated at or before each fit date. No feature search and no 2022-2023 test evaluation.\n\n'
    report+='## Later-2021 validation\n\n| Window | Method | Columns | MAE | RMSE | Bias |\n|---|---|---:|---:|---:|---:|\n'
    for row in summary.loc[summary.period.eq('later_validation')].itertuples():report+=f'| {row.window} | {row.method} | {row.feature_columns} | {row.MAE:.6f} | {row.RMSE:.6f} | {row.bias:.6f} |\n'
    report+='\nForward and Stepwise produced identical lists; their forecasts and scores are identical, not independent replications. Column counts include six weekday controls; ZIP FE are additional. Check horizon-specific results and paired differences, not only average ranks.\n\n'
    report+='The lists were selected on early 2021 with a 30-day model, so this asks whether those same lists remain useful at 60 days, not which features would be selected afresh at 60 days. Both parts of 2021 were examined in earlier work. This is exploratory sensitivity analysis and cannot establish a final independent winner. HAC tests and pointwise block CIs do not account fully for prior selection; Holm correction across 20 comparisons does not repair that bias. Positive delta MAE favors the candidate named in paired_comparisons.csv.\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    checks={'source_hash_matches_selection_run':True,'cohort_dates_targets_match_30d_archive':True,
        'features_reselected':False,'all_predictions_finite':bool(np.isfinite(predictions).all()),
        'forward_stepwise_predictions_identical':True,'cached_subset_OLS_matches_direct_OLS':True,
        'regions':n,'forecast_dates':v,'test_evaluated':False}
    (folder/'verification.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
    print(summary.to_string(index=False),flush=True);print('Saved to:',folder.resolve(),flush=True)
    return summary,metrics,paired,folder


In [2]:
SOURCE = Path.cwd() / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
SELECTION_FILE = Path.cwd() / 'individual_feature_selection_outputs' / 'three_methods_20261026T152516441253Z' / 'selected_features_by_method.json'
OUTPUT = Path.cwd() / 'individual_feature_selection_outputs'
summary, horizons, comparisons, RUN_FOLDER = run_frozen_window_comparison(SOURCE, OUTPUT, SELECTION_FILE)
display(summary.loc[summary.period.eq('later_validation')])
display(horizons.loc[horizons.period.eq('later_validation')])
print(RUN_FOLDER.resolve())


60-day frozen specification FULL fitted (34 candidate columns).
60-day frozen specification BACKWARD fitted (21 candidate columns).
60-day frozen specification FORWARD fitted (9 candidate columns).
60-day frozen specification STEPWISE fitted (9 candidate columns).
          period  window   method  feature_columns      MAE     RMSE      bias
later_validation      30 BACKWARD               21 1.178472 2.018709 -0.011035
later_validation      30  FORWARD                9 1.171625 2.013175 -0.038075
later_validation      30     FULL               34 1.180819 2.023986  0.021619
later_validation      30 STEPWISE                9 1.171625 2.013175 -0.038075
later_validation      60 BACKWARD               21 1.159535 2.003030 -0.022042
later_validation      60  FORWARD                9 1.164881 2.003248 -0.027737
later_validation      60     FULL               34 1.159097 2.008269 -0.000615
later_validation      60 STEPWISE                9 1.164881 2.003248 -0.027737
       selection      30

,period,window,method,feature_columns,MAE,RMSE,bias
0,later_validation,30,BACKWARD,21,1.178472,2.018709,-0.011035
1,later_validation,30,FORWARD,9,1.171625,2.013175,-0.038075
2,later_validation,30,FULL,34,1.180819,2.023986,0.021619
3,later_validation,30,STEPWISE,9,1.171625,2.013175,-0.038075
4,later_validation,60,BACKWARD,21,1.159535,2.003030,-0.022042
5,later_validation,60,FORWARD,9,1.164881,2.003248,-0.027737
6,later_validation,60,FULL,34,1.159097,2.008269,-0.000615
7,later_validation,60,STEPWISE,9,1.164881,2.003248,-0.027737


,period,window,method,horizon,feature_columns,regions,forecast_dates,MAE,RMSE,bias
24,later_validation,30,FULL,1,34,585,181,1.122244,1.866431,0.001613
25,later_validation,30,FULL,2,34,585,181,1.179853,2.021578,0.018675
26,later_validation,30,FULL,3,34,585,181,1.240359,2.183948,0.044568
27,later_validation,30,BACKWARD,1,21,585,181,1.124142,1.869372,-0.017230
28,later_validation,30,BACKWARD,2,21,585,181,1.177443,2.017722,-0.013553
29,later_validation,30,BACKWARD,3,21,585,181,1.233833,2.169032,-0.002323
30,later_validation,30,FORWARD,1,9,585,181,1.120859,1.866359,-0.037562
31,later_validation,30,FORWARD,2,9,585,181,1.170560,2.009157,-0.040547
32,later_validation,30,FORWARD,3,9,585,181,1.223455,2.164010,-0.036114
33,later_validation,30,STEPWISE,1,9,585,181,1.120859,1.866359,-0.037562


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\individual_feature_selection_outputs\frozen_window_60_20261026T154204961791Z


## Read first

* window_method_comparison_summary.csv: 30/60-day mean MAE, RMSE and bias.
* window_method_comparison_by_horizon.csv: days 1/2/3 individually.
* paired_comparisons.csv: paired date-level changes, pointwise 14-day block intervals, HAC/adjusted p-values. Positive delta favors the candidate.
* frozen_feature_lists.json and protocol.json: exact lists and data lineage.
* verification.json: source/cohort/forecast checks.

All p-values remain exploratory after earlier selection and validation inspection. Holm across this run's 20 comparisons is not a post-selection correction. Bootstrap resamples saved forecast errors, not the complete fitting/selection pipeline. No 2022-2023 test scores are generated.


## Optional report figure

Requires matplotlib. Saves a 300-dpi PNG and an SVG inside the result folder.


In [3]:
from pathlib import Path

def plot_window_comparison(folder):
    folder=Path(folder);dest=folder/'figures';dest.mkdir(exist_ok=True)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    s=pd.read_csv(folder/'window_method_comparison_summary.csv')
    s=s.loc[s.period.eq('later_validation')]
    order=['FULL','BACKWARD','FORWARD','STEPWISE']
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,metric in zip(axes,['MAE','RMSE']):
        for j,(window,color) in enumerate([(30,'#94a3b8'),(60,'#2563eb')]):
            values=s.loc[s.window.eq(window)].set_index('method').reindex(order)[metric]
            bars=ax.bar(np.arange(4)+(j-.5)*.36,values,width=.36,color=color,label=f'{window}-day window')
            ax.bar_label(bars,fmt='%.4f',fontsize=8,padding=4)
        ax.set_xticks(range(4),order);ax.set_title(f'Later-2021 {metric}');ax.set_ylabel('Recorded-onset cases')
        ax.set_ylim(0,s[metric].max()*1.2);ax.legend(frameon=False,fontsize=9)
        ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.suptitle('Frozen feature lists: 30 vs 60-day training windows')
    fig.text(.5,.015,'Same 585 regions, forecast dates and horizons 1/2/3 | Weekly refits | No feature reselection | Exploratory validation',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.06,1,.94])
    for ext in ['png','svg']:fig.savefig(dest/f'01_window_comparison.{ext}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    return dest


In [4]:
try:
    print(plot_window_comparison(RUN_FOLDER))
except ImportError:
    print('Optional plot requires matplotlib: %pip install matplotlib')


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\individual_feature_selection_outputs\frozen_window_60_20261026T154204961791Z\figures


## Read completed results without refitting

Run this cell independently when reviewing the completed run.


In [5]:
from pathlib import Path
import pandas as pd
EXISTING = Path.cwd() / 'individual_feature_selection_outputs' / 'frozen_window_60_20261026T153858733079Z'
display(pd.read_csv(EXISTING / 'window_method_comparison_summary.csv'))
print((EXISTING / 'RESULTS.md').read_text(encoding='utf-8'))


,period,window,method,feature_columns,MAE,RMSE,bias
0,later_validation,30,BACKWARD,21,1.178472,2.018709,-0.011035
1,later_validation,30,FORWARD,9,1.171625,2.013175,-0.038075
2,later_validation,30,FULL,34,1.180819,2.023986,0.021619
3,later_validation,30,STEPWISE,9,1.171625,2.013175,-0.038075
4,later_validation,60,BACKWARD,21,1.159535,2.003030,-0.022042
5,later_validation,60,FORWARD,9,1.164881,2.003248,-0.027737
6,later_validation,60,FULL,34,1.159097,2.008269,-0.000615
7,later_validation,60,STEPWISE,9,1.164881,2.003248,-0.027737
8,selection,30,BACKWARD,21,1.059605,1.808737,0.098616
9,selection,30,FORWARD,9,1.060438,1.810525,0.120204


# Frozen feature combinations: 30 vs 60-day windows

Feature membership is taken unchanged from the three-method Notebook 15 run. All 585 regions, forecast origins and horizons match the 30-day comparison. Only the rolling parameter training window changes to 60 days. Weekly refits still use only outcomes dated at or before each fit date. No feature search and no 2022-2023 test evaluation.

## Later-2021 validation

| Window | Method | Columns | MAE | RMSE | Bias |
|---|---|---:|---:|---:|---:|
| 30 | BACKWARD | 21 | 1.178472 | 2.018709 | -0.011035 |
| 30 | FORWARD | 9 | 1.171625 | 2.013175 | -0.038075 |
| 30 | FULL | 34 | 1.180819 | 2.023986 | 0.021619 |
| 30 | STEPWISE | 9 | 1.171625 | 2.013175 | -0.038075 |
| 60 | BACKWARD | 21 | 1.159535 | 2.003030 | -0.022042 |
| 60 | FORWARD | 9 | 1.164881 | 2.003248 | -0.027737 |
| 60 | FULL | 34 | 1.159097 | 2.008269 | -0.000615 |
| 60 | STEPWISE | 9 | 1.164881 | 2.003248 | -0.027737 |

Forward and Stepwise produced identical lists; their forec